-----

<div align="center">
<h1 style="color: #2c3e50;">V. Grafik Interfeys (PyQt5) va Chat Loyihasi</h1>
<p><i>Ushbu bo'limda biz tarmoq dasturlash bilimlarimizni zamonaviy grafik foydalanuvchi interfeysi (GUI) bilan birlashtiramiz.</i></p>
</div>

-----

### 5.1. PyQt5 GUI Dizayni: Signals & Slots

Tarmoq ilovalari uchun interfeys yaratishda PyQt5 ning **Signal-Slot** mexanizmi markaziy o'rin tutadi. Bu tizim hodisalarga (masalan, tugma bosilishi yoki xabar kelishi) asoslangan dasturlashni ta'minlaydi.

  * **Chat oynasi:** Xabarlarni ko'rsatish uchun `QTextEdit` va yozish uchun `QLineEdit` vidjetlaridan foydalaniladi.
  * **Signallar:** Tarmoqdan yangi xabar kelganda, maxsus signal (`pyqtSignal`) orqali interfeys yangilanadi.
  * **Kiberxavfsizlik:** Login formasida parollarni ochiq ko'rinishda saqlamaslik va kiritish maydonida `EchoMode`ni `Password` qilib sozlash zarur.

-----

### 5.2. Network-Worker Pattern: Oqimlarni ajratish

Tarmoq dasturlashda eng katta xato — tarmoq kodini (masalan, `recv()`) asosiy GUI oqimida (Main Thread) ishlatishdir.

  * **Muammo:** `recv()` metodining tabiatiga ko'ra, u ma'lumot kelguncha "blocking" (to'xtatib turuvchi) rejimda bo'ladi. Agar u asosiy oqimda bo'lsa, interfeys qotib qoladi (Not Responding).
  * **Yechim:** Tarmoq kodi alohida **Worker Thread** (`QThread`) ichida ishlashi kerak.

-----

### 5.3. Real-time Chat: Broadcasting (Xabarlarni tarqatish)

Haqiqiy chat dasturi uchun serverga **Broadcasting** algoritmi kerak. Ya'ni, bir mijozdan kelgan xabar barcha faol ulanishlarga (klientlarga) yuborilishi shart.

**Broadcasting Algoritmi:**

1.  Barcha faol mijoz soketlarini bitta ro'yxatda (`list`) saqlash.
2.  Yangi xabar kelganda, tsikl orqali ro'yxatdagi har bir soketga ma'lumotni yuborish.
3.  **Xavfsizlik:** Agar biror mijoz ulanishi uzilsa (`ConnectionReset`), uni darhol ro'yxatdan o'chirish lozim, aks holda server "o'lik" soketlarga ma'lumot yuborishga urinib, qulab tushishi mumkin.

-----

### Amaliy Kod: PyQt5 Chat Klienti (Soddalashtirilgan Model)

Ushbu kodni Jupyter Notebook-da ishlatsangiz, alohida oyna ochiladi. (Kompyuteringizda `pip install PyQt5` o'rnatilgan bo'lishi shart).

```python
from PyQt5.QtWidgets import QApplication, QMainWindow, QTextEdit, QLineEdit, QVBoxLayout, QWidget
from PyQt5.QtCore import QThread, pyqtSignal
import socket

# Tarmoq ishlarini bajaruvchi alohida oqim
class ReceiverThread(QThread):
    message_received = pyqtSignal(str)

    def run(self):
        client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        client.connect(('127.0.0.1', 5050))
        while True:
            try:
                data = client.recv(1024).decode('utf-8')
                if data:
                    self.message_received.emit(data)
            except:
                break

class ChatGUI(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Real-time Chat")
        self.display = QTextEdit(self)
        self.input = QLineEdit(self)
        
        layout = QVBoxLayout()
        layout.addWidget(self.display)
        layout.addWidget(self.input)
        
        container = QWidget()
        container.setLayout(layout)
        self.setCentralWidget(container)

        # Worker Threadni ishga tushirish
        self.thread = ReceiverThread()
        self.thread.message_received.connect(self.display.append)
        self.thread.start()

# Dasturni ishga tushirish qismi
# app = QApplication([])
# gui = ChatGUI()
# gui.show()
# app.exec_()
```

-----
